# 04) Model Analysis & Results

This file analyzes the saved baseline and LSTM results.

It does not retrain the model.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    roc_curve,
    confusion_matrix,
)
from sklearn.calibration import calibration_curve

In [ ]:
PROJECT_ROOT = Path("..").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ML_DIR = PROCESSED_DIR / "ml"
LSTM_DIR = PROCESSED_DIR / "lstm"
ANALYSIS_DIR = PROCESSED_DIR / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

with open(ML_DIR / "baseline_metrics.json", "r", encoding="utf-8") as f:
    baseline_metrics = json.load(f)

with open(LSTM_DIR / "test_metrics.json", "r", encoding="utf-8") as f:
    lstm_metrics = json.load(f)

with open(LSTM_DIR / "decision_threshold.json", "r", encoding="utf-8") as f:
    threshold_data = json.load(f)

training_history = pd.read_csv(
    LSTM_DIR / "training_history.csv"
)

lstm_predictions = pd.read_csv(
    LSTM_DIR / "test_predictions_by_timestep.csv"
)

baseline_predictions = pd.read_csv(
    ML_DIR / "baseline_test_predictions_by_timestep.csv"
)

per_timestep = pd.read_csv(
    LSTM_DIR / "test_metrics_by_timestep.csv"
)

detection_by_step = pd.read_csv(
    LSTM_DIR / "detection_rate_by_timestep.csv"
)

first_detection = pd.read_csv(
    LSTM_DIR / "first_detection_by_sequence.csv"
)

print("Baseline test:", baseline_metrics["test"])
print("LSTM test:", lstm_metrics)
print("LSTM threshold:", threshold_data["threshold"])

# 1) Overall Metrics

In [ ]:
overall_metrics = pd.DataFrame([
    {
        "model": "Logistic Regression",
        **baseline_metrics["test"],
    },
    {
        "model": "Causal LSTM",
        **lstm_metrics,
    },
])

print(overall_metrics)
overall_metrics.to_csv(
    ANALYSIS_DIR / "overall_metrics.csv",
    index=False,
)

# 2) Training History

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    training_history["epoch"],
    training_history["train_loss"],
    marker="o",
)
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("LSTM Training Loss")
plt.grid(True)
plt.tight_layout()
plt.savefig(
    ANALYSIS_DIR / "training_loss.png",
    dpi=150,
)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    training_history["epoch"],
    training_history["validation_pr_auc"],
    marker="o",
)
plt.xlabel("Epoch")
plt.ylabel("Validation PR-AUC")
plt.title("Validation PR-AUC by Epoch")
plt.grid(True)
plt.tight_layout()
plt.savefig(
    ANALYSIS_DIR / "validation_pr_auc.png",
    dpi=150,
)
plt.show()

# 3) Precision-Recall Curve

The curves use the same test prediction steps.

In [ ]:
y_lstm = lstm_predictions["target"].to_numpy()
p_lstm = lstm_predictions["risk_probability"].to_numpy()

y_base = baseline_predictions["target"].to_numpy()
p_base = baseline_predictions["risk_probability"].to_numpy()

precision_lstm, recall_lstm, _ = precision_recall_curve(
    y_lstm,
    p_lstm,
)
precision_base, recall_base, _ = precision_recall_curve(
    y_base,
    p_base,
)

plt.figure(figsize=(8, 5))
plt.plot(
    recall_base,
    precision_base,
    label=f"Logistic Regression (AP={average_precision_score(y_base, p_base):.3f})",
)
plt.plot(
    recall_lstm,
    precision_lstm,
    label=f"Causal LSTM (AP={average_precision_score(y_lstm, p_lstm):.3f})",
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Test Precision-Recall Curves")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(
    ANALYSIS_DIR / "test_precision_recall.png",
    dpi=150,
)
plt.show()

# 4) ROC Curve

In [ ]:
fpr_base, tpr_base, _ = roc_curve(y_base, p_base)
fpr_lstm, tpr_lstm, _ = roc_curve(y_lstm, p_lstm)

plt.figure(figsize=(8, 5))
plt.plot(
    fpr_base,
    tpr_base,
    label=f"Logistic Regression (ROC-AUC={roc_auc_score(y_base, p_base):.3f})",
)
plt.plot(
    fpr_lstm,
    tpr_lstm,
    label=f"Causal LSTM (ROC-AUC={roc_auc_score(y_lstm, p_lstm):.3f})",
)
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Test ROC Curves")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(
    ANALYSIS_DIR / "test_roc.png",
    dpi=150,
)
plt.show()

# 5) Risk Probability Distribution

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(
    lstm_predictions.loc[
        lstm_predictions["target"] == 0,
        "risk_probability"
    ],
    bins=30,
    alpha=0.7,
    label="Target 0",
)
plt.hist(
    lstm_predictions.loc[
        lstm_predictions["target"] == 1,
        "risk_probability"
    ],
    bins=30,
    alpha=0.7,
    label="Target 1",
)
plt.xlabel("Risk Probability")
plt.ylabel("Number of Checkpoints")
plt.title("LSTM Risk Probability Distribution")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(
    ANALYSIS_DIR / "risk_probability_distribution.png",
    dpi=150,
)
plt.show()

# 6) Calibration

In [ ]:
fraction_pos, mean_pred = calibration_curve(
    y_lstm,
    p_lstm,
    n_bins=10,
    strategy="uniform",
)

plt.figure(figsize=(8, 5))
plt.plot(
    mean_pred,
    fraction_pos,
    marker="o",
)
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Observed Positive Fraction")
plt.title("LSTM Calibration Curve")
plt.grid(True)
plt.tight_layout()
plt.savefig(
    ANALYSIS_DIR / "lstm_calibration.png",
    dpi=150,
)
plt.show()

# 7) Performance by Timestep

In [ ]:
print(per_timestep)
per_timestep.to_csv(
    ANALYSIS_DIR / "performance_by_timestep.csv",
    index=False,
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    per_timestep["time_step"],
    per_timestep["pr_auc"],
    marker="o",
)
plt.xlabel("Time Step")
plt.ylabel("PR-AUC")
plt.title("LSTM PR-AUC by Timestep")
plt.grid(True)
plt.tight_layout()
plt.savefig(
    ANALYSIS_DIR / "pr_auc_by_timestep.png",
    dpi=150,
)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    per_timestep["time_step"],
    per_timestep["f1"],
    marker="o",
)
plt.xlabel("Time Step")
plt.ylabel("F1")
plt.title("LSTM F1 by Timestep")
plt.grid(True)
plt.tight_layout()
plt.savefig(
    ANALYSIS_DIR / "f1_by_timestep.png",
    dpi=150,
)
plt.show()

# 8) Early Detection Rate

In [ ]:
print(detection_by_step)
detection_by_step.to_csv(
    ANALYSIS_DIR / "detection_rate_by_timestep.csv",
    index=False,
)

plt.figure(figsize=(8, 5))
plt.plot(
    detection_by_step["time_step"],
    detection_by_step["detection_rate"],
    marker="o",
)
plt.xlabel("Time Step")
plt.ylabel("Detection Rate")
plt.title("Positive Test Sequences Detected by Timestep")
plt.ylim(0, 1)
plt.grid(True)
plt.tight_layout()
plt.savefig(
    ANALYSIS_DIR / "detection_rate.png",
    dpi=150,
)
plt.show()

# 9) First Detection Distribution

In [ ]:
detected_only = first_detection[
    first_detection["first_detected_step"].notna()
]

print("Positive test sequences:", len(first_detection))
print("Detected:", len(detected_only))
print(
    "Detection rate:",
    len(detected_only) / len(first_detection)
    if len(first_detection) > 0 else np.nan
)
print(
    "Median detection step:",
    detected_only["first_detected_step"].median()
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(
    detected_only["first_detected_step"],
    bins=np.arange(0.5, int(detected_only["first_detected_step"].max()) + 1.5),
    rwidth=0.8,
)
plt.xlabel("First Detection Timestep")
plt.ylabel("Number of Positive Sequences")
plt.title("Distribution of First Detection Timestep")
plt.grid(True)
plt.tight_layout()
plt.savefig(
    ANALYSIS_DIR / "first_detection_distribution.png",
    dpi=150,
)
plt.show()

# 10) Confusion Matrix at the Selected Threshold

In [ ]:
selected_threshold = threshold_data["threshold"]
lstm_pred = (p_lstm >= selected_threshold).astype(int)

cm = confusion_matrix(y_lstm, lstm_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"],
)

print(cm_df)
cm_df.to_csv(
    ANALYSIS_DIR / "lstm_confusion_matrix.csv"
)

# 11) Module-level Analysis

In [ ]:
module_analysis = []

for module, part in lstm_predictions.groupby("code_module"):
    if part["target"].nunique() < 2:
        continue

    module_pred = (
        part["risk_probability"] >= selected_threshold
    ).astype(int)

    module_analysis.append({
        "code_module": module,
        "n_checkpoints": len(part),
        "positive_ratio": float(part["target"].mean()),
        "pr_auc": average_precision_score(
            part["target"],
            part["risk_probability"],
        ),
        "roc_auc": roc_auc_score(
            part["target"],
            part["risk_probability"],
        ),
        "precision": precision_score(
            part["target"], module_pred, zero_division=0
        ),
        "recall": recall_score(
            part["target"], module_pred, zero_division=0
        ),
        "f1": f1_score(
            part["target"], module_pred, zero_division=0
        ),
    })

module_analysis_df = pd.DataFrame(module_analysis).sort_values("code_module")
print(module_analysis_df)
module_analysis_df.to_csv(
    ANALYSIS_DIR / "module_analysis.csv",
    index=False,
)

# 12) Final Analysis Export

In [ ]:
analysis_summary = {
    "baseline_test": baseline_metrics["test"],
    "lstm_test": lstm_metrics,
    "lstm_threshold": selected_threshold,
    "best_validation_pr_auc": float(
        training_history["validation_pr_auc"].max()
    ),
    "positive_test_sequences": int(len(first_detection)),
    "detected_positive_sequences": int(len(detected_only)),
    "detection_rate": (
        float(len(detected_only) / len(first_detection))
        if len(first_detection) > 0 else np.nan
    ),
    "median_first_detection_step": (
        float(detected_only["first_detected_step"].median())
        if len(detected_only) > 0 else np.nan
    ),
}

with open(
    ANALYSIS_DIR / "analysis_summary.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(analysis_summary, f, indent=2)

print(json.dumps(analysis_summary, indent=2))

# ✅ What happened in this file?

We analyzed the saved training history, baseline, LSTM test predictions, performance by timestep, calibration, first detection timing, and module-level behavior.

This file is for interpreting the trained model; it does not change or retrain the model.